# Wine dataset
- Show chemical composition of wine
- Has 3 classes

In [64]:
# Load the data
from sklearn.datasets import load_wine
import pandas as pd

dataset = load_wine()

X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
y = dataset.target

In [66]:
# display the first 10 data points X and the labels y

print(X[:10])
print(y[:10])

   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   
3    14.37        1.95  2.50               16.8      113.0           3.85   
4    13.24        2.59  2.87               21.0      118.0           2.80   
5    14.20        1.76  2.45               15.2      112.0           3.27   
6    14.39        1.87  2.45               14.6       96.0           2.50   
7    14.06        2.15  2.61               17.6      121.0           2.60   
8    14.83        1.64  2.17               14.0       97.0           2.80   
9    13.86        1.35  2.27               16.0       98.0           2.98   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.

In [67]:
# size of data

print(X.shape) # rows, columns
print(y.shape)

(178, 13)
(178,)


In [68]:
# value counts of y

print(pd.Series(y).value_counts())

# seems balanced

1    71
0    59
2    48
Name: count, dtype: int64


# Find the best model + best hyperparameter
- I want to use other models with different hyperparameters and find the best model

In [54]:
# Compare Multiple Models (Cross-Validation)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "SVM (RBF Kernel)": SVC(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier()
}

for name, model in models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", model)])
    scores = cross_val_score(pipe, X, y, cv=5)
    print(f"{name}: {scores.mean():.4f}")

Logistic Regression: 0.9832
SVM (RBF Kernel): 0.9833
Random Forest: 0.9610
Gradient Boosting: 0.9330


### SVC gave good results. So, perform hyperparameter tuning of SVC

In [58]:
# Hyperparameter Tuning (Example: SVM)
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC())
])

# specify range of hyperparameters that you want to check: Total HP = 3x3x1 = 9 
param_grid = {
    "svc__C": [0.1, 1, 10],
    "svc__gamma": ["scale", 0.1, 0.01],
    "svc__kernel": ["rbf"]
}

gs = GridSearchCV(pipe, param_grid, cv=5, n_jobs=-1)
gs.fit(X_train, y_train)

print("Best HP:", gs.best_params_)

Best HP: {'svc__C': 1, 'svc__gamma': 0.01, 'svc__kernel': 'rbf'}


In [59]:
# Retrieve the best model
best_model = gs

In [73]:
# Lets check the prediction on 1 data point, say index = 3
# ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium',
#        'total_phenols', 'flavanoids', 'nonflavanoid_phenols',
#        'proanthocyanins', 'color_intensity', 'hue',
       # 'od280/od315_of_diluted_wines', 'proline']
sample = [[14.37,  1.95,  2.50, 16.8, 113.0, 3.85, 3.49,  0.24, 2.18,\
           7.80,  0.86, 3.45,  1480.0 ],]

sample_df = pd.DataFrame(sample, columns=X.columns)

prediction = best_model.predict(sample_df)

print("Prediction:", prediction)

Prediction: [0]


In [74]:
y_pred = best_model.predict(X_test)

In [75]:
# lets see the first 6 test data points and their prediction
print("First 6 test data :\n", X_test[:6])
print("First 6 prediction:",   y_pred[:6])

First 6 test data :
      alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
10     14.10        2.16  2.30               18.0      105.0           2.95   
134    12.51        1.24  2.25               17.5       85.0           2.00   
28     13.87        1.90  2.80               19.4      107.0           2.95   
121    11.56        2.05  3.23               28.5      119.0           3.18   
62     13.67        1.25  1.92               18.0       94.0           2.10   
51     13.83        1.65  2.60               17.2       94.0           2.45   

     flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
10         3.32                  0.22             2.38             5.75  1.25   
134        0.58                  0.60             1.25             5.45  0.75   
28         2.97                  0.37             1.76             4.50  1.25   
121        5.08                  0.47             1.87             6.00  0.93   
62         1.79     

In [76]:
# Evaluate the Best Model
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      1.00      1.00        14
           2       1.00      1.00      1.00        10

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36

[[12  0  0]
 [ 0 14  0]
 [ 0  0 10]]


Recommended advanced tasks

Some add-ons for ML:

✔ PCA + Classification

Reduce dimension → classify with SVM.

✔ Plot decision boundaries (2D PCA)
✔ Use XGBoost classifier

(XGBoost performs extremely well on the Wine dataset)

✔ Explain Model (SHAP values)
✔ Create an ML Pipeline API with FastAPI / Django